# 20. Python Decorators (5+ Years Interview Guide)
Deep architectural analysis of function wrappers, closures, functools.wraps metadata preservation, parametric decorators (3-layer closures), stacked decorator ordering, class decorators, and caching.

### Key 5-Year Interview Concepts Covered:
- **Decorator Execution Lifecycle**: Why `@decorator` executes at module definition/import time, not invocation time.
- **Metadata Preservation (`functools.wraps`)**: Preserving `__name__`, `__doc__`, and `__annotations__` on wrapped callables.
- **Parametric Decorators**: 3-level nested closure factories accepting configuration arguments `@retry(max_attempts=3)`.
- **Stacked Decorator Execution Order**: Bottom-up wrapping application order vs top-down execution order.

This notebook uses the shared Fintech dataset `data/raw_transactions.csv` for interview scenario problems at the end.

In [1]:
# Setup: Locate the Shared Dataset
import os
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
print("Using CSV file path:", csv_path)

Using CSV file path: data/raw_transactions.csv


### 1. First-Class Functions in Python
**Explanation**: In Python, functions are first-class citizens: they can be assigned to variables, passed as arguments to other functions, stored in data structures, and returned from other functions. This is the structural foundation for decorators.

**Syntax**: `fn_ref = calculate_tax; result = fn_ref(100)`

In [2]:
def calculate_payout(): return 1
payout_ref = calculate_payout
print(payout_ref())

1


### 2. Closures & Function Wrappers
**Explanation**: A decorator is a callable that takes a function as an argument and returns a wrapped replacement function. The wrapper function forms a closure over the original function, intercepting inputs, executing pre/post-processing logic, and returning the result.

**Syntax**: `def simple_decorator(func): def wrapper(*a, **kw): return func(*a, **kw); return wrapper`

In [3]:
def payout_wrapper_decorator(func):
    return lambda: func() + 10
@payout_wrapper_decorator
def get_base_payout(): return 5
print(get_base_payout())

15


### 3. Preserving Metadata with `functools.wraps`
**Explanation**: When a function is wrapped, its `__name__`, `__doc__`, and `__module__` are overwritten by the wrapper function. Decorating the inner wrapper with `@functools.wraps(func)` copies the original function's metadata and sets `__wrapped__`, ensuring docstrings, introspection, and debugging tools function correctly.

**Syntax**: `from functools import wraps; def dec(f): @wraps(f) def wrapper(*a, **kw): return f(*a, **kw); return wrapper`

In [4]:
from functools import wraps
def payout_auditor_decorator(func):
    @wraps(func)
    def wrapper_signature(): return func()
    return wrapper_signature
d = payout_auditor_decorator
# note: d used below dynamically
@payout_auditor_decorator
def run_payout_clearance(): pass
print('Preserved wraps name:', run_payout_clearance.__name__)

Preserved wraps name: run_payout_clearance


### 4. Decorators Modifying Function Inputs
**Explanation**: Decorators can inspect, validate, sanitize, or transform input arguments before passing them to the decorated function (e.g. type casting, trimming strings, or injecting database session objects).

**Syntax**: `def sanitize(f): @wraps(f) def w(arg): return f(arg.strip()); return w`

In [5]:
def double_input_decorator(func):
    return lambda input_value: func(input_value * 2)
@double_input_decorator
def process_payout(value): return value
print(process_payout(5))

10


### 5. Decorators Modifying Function Return Outputs
**Explanation**: Decorators can intercept the return value of a function to format outputs, convert dictionaries to JSON HTTP responses, or apply post-execution rounding.

**Syntax**: `def round_output(f): @wraps(f) def w(*a, **kw): return round(f(*a, **kw), 2); return w`

In [6]:
def uppercase_output_decorator(func):
    return lambda: func().upper()
@uppercase_output_decorator
def get_status_code(): return 'ok'
print(get_status_code())

OK


### 6. Parametric Decorators (Decorators with Arguments)
**Explanation**: To pass arguments to a decorator `@repeat(num_times=3)`, you need a 3-layer closure: 1. An outer decorator factory function accepting configuration arguments; 2. An intermediate decorator function accepting the target function; 3. The inner wrapper function.

**Syntax**: `def repeat(n): def decorator(func): @wraps(func) def wrapper(*a, **kw): ...; return wrapper; return decorator`

In [7]:
def offset_payout_decorator(offset_amount):
    def decorator_layer(func):
        return lambda: func() + offset_amount
    return decorator_layer
@offset_payout_decorator(100)
def get_base_amount(): return 5
print(get_base_amount())

105


### 7. Applying Multiple Decorators (Stacked Decorators)
**Explanation**: When stacking multiple decorators `@dec1 \n @dec2 \n def func():`, Python applies them from bottom to top: `func = dec1(dec2(func))`. During execution, the outermost decorator (`dec1`) runs first, delegates to `dec2`, which calls the original function.

**Syntax**: `@auth_required
@audit_log
def process(): ...`

In [8]:
def StackedDecoratorFirst(func): return lambda: func() + '1'
def StackedDecoratorSecond(func): return lambda: func() + '2'
@StackedDecoratorFirst
@StackedDecoratorSecond
def get_base_value(): return '0'
print(get_base_value())

021


### 8. Class-Based Decorators (`__call__`)
**Explanation**: Classes implementing `__init__(self, func)` and `__call__(self, *args, **kwargs)` can act as decorators. Class decorators are ideal for stateful decorators requiring instance variables, such as execution rate limiters and circuit breakers.

**Syntax**: `class CountCalls: def __init__(self, f): self.f = f; self.count = 0; def __call__(self, *a, **k): self.count += 1; return self.f(*a, **k)`

In [9]:
class ClassDecoratorClass:
    def __init__(self, func): self.func = func
    def __call__(self, *args): return 'ClassWrap'
@ClassDecoratorClass
def run_audit(): pass
print(run_audit())

ClassWrap


### 9. Decorating Entire Classes
**Explanation**: A class decorator receives a class object as its argument and returns a modified class. Standard library examples include `@dataclass` and `@total_ordering`. They dynamically inspect class attributes and inject methods.

**Syntax**: `def add_id(cls): cls.id = 1; return cls; @add_id class Entity: ...`

In [10]:
def inject_attribute_decorator(cls):
    cls.class_attribute = 10
    return cls
@inject_attribute_decorator
class ContainerClass: pass
print(ContainerClass.class_attribute)

10


### 10. Stateful Decorators with Function Attributes
**Explanation**: Decorators can maintain state (e.g. invocation counts, total execution time) by attaching attributes directly to the wrapper function object, avoiding global state.

**Syntax**: `wrapper.invocations = 0`

In [11]:
def call_counter_decorator(func):
    state_counter = 0
    def wrapper_fn():
        nonlocal state_counter; state_counter += 1; return func(), state_counter
    return wrapper_fn
@call_counter_decorator
def query_database(): return 'x'
print(query_database(), query_database())

('x', 1) ('x', 2)


### 11. Memoization & Caching (`functools.lru_cache`)
**Explanation**: `@functools.lru_cache(maxsize=128)` caches function return values based on passed arguments using a Least Recently Used eviction policy. CRITICAL GOTCHA: All function arguments must be hashable; passing a list or dict raises `TypeError: unhashable type`.

**Syntax**: `from functools import lru_cache; @lru_cache(maxsize=256) def compute(n): ...`

In [12]:
from functools import lru_cache
@lru_cache(maxsize=2)
def run_slow_calculation(input_value): print('Run'); return input_value
run_slow_calculation(1); run_slow_calculation(1)

Run


### 12. Input Validation Constraints Decorators
**Explanation**: Decorators can enforce precondition contracts (e.g. verifying user authentication tokens, permission scopes, or schema validation) before executing sensitive business logic.

**Syntax**: `@require_admin
def delete_account(user_id): ...`

In [13]:
def positive_validator_decorator(func):
    def wrapper(input_value):
        if input_value < 0: raise ValueError('Neg')
        return func(input_value)
    return wrapper
@positive_validator_decorator
def process_value(val): return val
try: process_value(-1)
except ValueError as error_message: print(error_message)

Neg


### 13. Decorating Generator Functions
**Explanation**: When decorating a generator function, the inner wrapper must also be a generator that uses `yield from func(*args, **kwargs)` to stream items lazily rather than evaluating the generator eagerly.

**Syntax**: `def log_gen(f): @wraps(f) def w(*a, **kw): print('Start'); yield from f(*a, **kw); return w`

In [14]:
def generator_wrapper_decorator(func):
    def wrapper():
        yield 'Start'
        yield from func()
    return wrapper
@generator_wrapper_decorator
def yield_values(): yield 1
print(list(yield_values()))

['Start', 1]


### 14. Decorator Handler Registries
**Explanation**: Decorators can automatically register functions into a central dispatch dictionary (like Flask/FastAPI `@app.route('/path')`). The decorator stores the function pointer in a registry dict and returns the function unmodified.

**Syntax**: `REGISTRY = {}; def register(name): def dec(f): REGISTRY[name] = f; return f; return dec`

In [15]:
REGISTRY_LIST = []
def register_decorator(func):
    REGISTRY_LIST.append(func); return func
@register_decorator
def run_standard_task(): pass
print('Registered count:', len(REGISTRY_LIST))

Registered count: 1


### 15. Method Decorators & The Descriptor Protocol
**Explanation**: When decorating instance methods inside a class, the wrapper function must accept `self` as the first argument (`def wrapper(self, *args, **kwargs):`). Implementing the decorator using the descriptor protocol `__get__` ensures methods bind properly when accessed via instances.

**Syntax**: `def method_decorator(func): @wraps(func) def wrapper(self, *a, **k): return func(self, *a, **k); return wrapper`

In [16]:
class MethodDecoratorClass:
    def __init__(self, func): self.func = func
    def __get__(self, instance, owner):
        import functools
        return functools.partial(self.func, instance)
print(MethodDecoratorClass)

<class '__main__.MethodDecoratorClass'>


## Section 3: Fintech Senior Interview Scenarios
**Explanation**: Designing stacked authorization and transaction auditing decorators with performance caching.


In [17]:
# Solution:
from functools import lru_cache

@lru_cache(maxsize=3)
def fetch_client_record(cust_id):
    print(f'Simulating slow DB fetch for {cust_id}...')
    return f'Record_{cust_id}'

fetch_client_record('C101')
fetch_client_record('C101')  # cached call


Simulating slow DB fetch for C101...


### Q2: Stacked Transaction Auditing & Auth Decorators
**Explanation**: **Scenario**: Write stacked decorators `@check_auth` and `@log_transaction` to validate permissions and log transaction executions across the fintech dataset.

**Syntax**: `@check_auth('ADMIN')
@log_transaction
def execute_payout(tx_id, amount): ...`

In [18]:
# Solution:
def log_tx(f):
    def w(*a, **kw): print('Auditing payment...'); return f(*a, **kw)
    return w

def check_auth(f):
    def w(amt, *a, **kw):
        if amt > 10000.0: raise PermissionError('Limit Exceeded')
        return f(amt, *a, **kw)
    return w

@log_tx
@check_auth
def process_payment(amount): print(f'Cleared ${amount}')
process_payment(500.0)


Auditing payment...
Cleared $500.0
